# Fase 3 — Benchmarks e Coleta de Dados
**Tema:** Reduções e Prefix-Sum (Scan) Paralelos  
**Grupo:** Gustavo Santos Oliveira · Olívier Queirós Pereira · Ruan Dener Souza Silva  

Este notebook realiza a medição sistemática de desempenho de todas as implementações CPU e CUDA do projeto, calculando tempo médio, desvio padrão, speedup e eficiência para cada configuração de entrada.

## Célula 1 — Informações do ambiente

In [ ]:
# Registra GPU e versão CUDA — SEMPRE rode esta célula primeiro e salve o output
!nvidia-smi
print()
!nvcc --version

## Célula 2 — Instalação de dependências

In [ ]:
!pip install numba numpy pandas matplotlib seaborn -q

## Célula 3 — Imports e configuração global

In [ ]:
import numpy as np
import pandas as pd
import time
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from numba import cuda, int32

# Reprodutibilidade
np.random.seed(42)

# Diretório de saída
os.makedirs('resultados', exist_ok=True)

# Configurações de benchmark
REPETICOES     = 7       # número de execuções por configuração (descarta min e max)
TAMANHOS       = [1_000, 10_000, 100_000, 1_000_000, 10_000_000]
THREADS_BLOCO  = [32, 64, 128, 256, 512]   # usado nas kernels multi-bloco

print('Ambiente configurado.')
print(f'Tamanhos de entrada : {TAMANHOS}')
print(f'Repetições por config: {REPETICOES} (descarta menor e maior → média de {REPETICOES-2})')

## Célula 4 — Implementações CPU (baseline)

In [ ]:
# ── CPU sequencial ────────────────────────────────────────────────────────────

def soma_cpu(v):
    soma = 0
    for i in range(len(v)):
        soma += v[i]
    return soma

def maximo_cpu(v):
    maior = v[0]
    for i in range(1, len(v)):
        if v[i] > maior:
            maior = v[i]
    return maior

def minimo_cpu(v):
    menor = v[0]
    for i in range(1, len(v)):
        if v[i] < menor:
            menor = v[i]
    return menor

def scan_cpu(v):
    """Prefix-sum inclusivo sequencial."""
    out = np.empty_like(v)
    acc = 0
    for i in range(len(v)):
        acc += v[i]
        out[i] = acc
    return out

# Sanidade
v_teste = np.array([1, 2, 3, 4, 5], dtype=np.int32)
assert soma_cpu(v_teste) == 15
assert maximo_cpu(v_teste) == 5
assert minimo_cpu(v_teste) == 1
assert list(scan_cpu(v_teste)) == [1, 3, 6, 10, 15]
print('CPU OK — soma=15, max=5, min=1, scan=[1,3,6,10,15]')

## Célula 5 — Kernels CUDA (redução ingênua)

In [ ]:
# ── Redução ingênua com divergência de warp ───────────────────────────────────
# Fonte: src/cuda/reduction_naive.py

BLOCO_NAIVE = 512   # bloco padrão para kernels ingênuas

@cuda.jit
def soma_ingenua_kernel(v_in, v_out):
    sdata = cuda.shared.array(shape=(512,), dtype=int32)
    tid = cuda.threadIdx.x
    bid = cuda.blockIdx.x
    idx = bid * cuda.blockDim.x + tid
    sdata[tid] = v_in[idx] if idx < v_in.size else 0
    cuda.syncthreads()
    stride = 1
    while stride < cuda.blockDim.x:
        if tid % (2 * stride) == 0 and tid + stride < cuda.blockDim.x:
            sdata[tid] += sdata[tid + stride]
        stride *= 2
        cuda.syncthreads()
    if tid == 0:
        v_out[bid] = sdata[0]

@cuda.jit
def maximo_ingenuo_kernel(v_in, v_out):
    sdata = cuda.shared.array(shape=(512,), dtype=int32)
    tid = cuda.threadIdx.x
    bid = cuda.blockIdx.x
    idx = bid * cuda.blockDim.x + tid
    sdata[tid] = v_in[idx] if idx < v_in.size else -2147483648
    cuda.syncthreads()
    stride = 1
    while stride < cuda.blockDim.x:
        if tid % (2 * stride) == 0 and tid + stride < cuda.blockDim.x:
            sdata[tid] = max(sdata[tid], sdata[tid + stride])
        stride *= 2
        cuda.syncthreads()
    if tid == 0:
        v_out[bid] = sdata[0]

@cuda.jit
def minimo_ingenuo_kernel(v_in, v_out):
    sdata = cuda.shared.array(shape=(512,), dtype=int32)
    tid = cuda.threadIdx.x
    bid = cuda.blockIdx.x
    idx = bid * cuda.blockDim.x + tid
    sdata[tid] = v_in[idx] if idx < v_in.size else 2147483647
    cuda.syncthreads()
    stride = 1
    while stride < cuda.blockDim.x:
        if tid % (2 * stride) == 0 and tid + stride < cuda.blockDim.x:
            sdata[tid] = min(sdata[tid], sdata[tid + stride])
        stride *= 2
        cuda.syncthreads()
    if tid == 0:
        v_out[bid] = sdata[0]

def reducao_ingenua(v_gpu, op='soma', threads=BLOCO_NAIVE):
    """Executa a redução ingênua em múltiplos blocos até resultado único."""
    n = v_gpu.size
    blocos = (n + threads - 1) // threads
    parcial = cuda.device_array(blocos, dtype=np.int32)
    if op == 'soma':
        soma_ingenua_kernel[blocos, threads](v_gpu, parcial)
    elif op == 'max':
        maximo_ingenuo_kernel[blocos, threads](v_gpu, parcial)
    else:
        minimo_ingenuo_kernel[blocos, threads](v_gpu, parcial)
    # redução final em CPU se necessário
    h = parcial.copy_to_host()
    if op == 'soma':  return int(h.sum())
    if op == 'max':   return int(h.max())
    return int(h.min())

print('Kernels de redução ingênua compiladas.')

## Célula 6 — Kernel CUDA: Hillis-Steele (scan inclusivo)

In [ ]:
# ── Hillis-Steele — scan inclusivo ────────────────────────────────────────────
# Fonte: src/cuda/hillis_steele.py

HS_BLOCO = 512

@cuda.jit
def hillis_steele_kernel(v_in, v_out):
    temp = cuda.shared.array(shape=(512,), dtype=int32)
    tid  = cuda.threadIdx.x
    temp[tid] = v_in[tid] if tid < v_in.size else 0
    cuda.syncthreads()
    stride = 1
    while stride < cuda.blockDim.x:
        valor = 0
        if tid >= stride:
            valor = temp[tid - stride]
        cuda.syncthreads()
        if tid >= stride:
            temp[tid] += valor
        cuda.syncthreads()
        stride *= 2
    if tid < v_out.size:
        v_out[tid] = temp[tid]

def scan_hillis_steele(v_gpu):
    n   = v_gpu.size
    out = cuda.device_array(n, dtype=np.int32)
    hillis_steele_kernel[1, HS_BLOCO](v_gpu, out)
    return out.copy_to_host()

print('Kernel Hillis-Steele compilada.')

## Célula 7 — Kernel CUDA: Blelloch (scan exclusivo)

In [ ]:
# ── Blelloch — scan exclusivo work-efficient ──────────────────────────────────
# Fonte: src/cuda/blelloch.py

BL_BLOCO = 512

@cuda.jit
def blelloch_scan_kernel(v_in, v_out):
    sdata = cuda.shared.array(shape=(512,), dtype=int32)
    tid   = cuda.threadIdx.x
    sdata[tid] = v_in[tid] if tid < v_in.size else 0
    cuda.syncthreads()
    # Fase up-sweep (redução)
    offset = 1
    d = BL_BLOCO // 2
    while d > 0:
        cuda.syncthreads()
        if tid < d:
            ai = offset * (2 * tid + 1) - 1
            bi = offset * (2 * tid + 2) - 1
            sdata[bi] += sdata[ai]
        offset *= 2
        d //= 2
    cuda.syncthreads()
    if tid == 0:
        sdata[BL_BLOCO - 1] = 0
    # Fase down-sweep
    d = 1
    while d < BL_BLOCO:
        offset //= 2
        cuda.syncthreads()
        if tid < d:
            ai = offset * (2 * tid + 1) - 1
            bi = offset * (2 * tid + 2) - 1
            temp       = sdata[ai]
            sdata[ai]  = sdata[bi]
            sdata[bi] += temp
        d *= 2
    cuda.syncthreads()
    if tid < v_out.size:
        v_out[tid] = sdata[tid]

def scan_blelloch(v_gpu):
    n   = v_gpu.size
    out = cuda.device_array(n, dtype=np.int32)
    blelloch_scan_kernel[1, BL_BLOCO](v_gpu, out)
    return out.copy_to_host()

print('Kernel Blelloch compilada.')

## Célula 8 — Validação de corretude (OBRIGATÓRIO antes do benchmark)

In [ ]:
# ── Valida cada kernel contra a referência CPU ────────────────────────────────
# Nunca meça tempo de kernel incorreta!

v_val   = np.arange(1, 513, dtype=np.int32)   # 512 elementos
d_val   = cuda.to_device(v_val)
ref_sum = int(v_val.sum())
ref_max = int(v_val.max())
ref_min = int(v_val.min())
ref_scan_inc = np.cumsum(v_val).astype(np.int32)       # scan inclusivo
ref_scan_exc = np.concatenate([[0], np.cumsum(v_val)[:-1]]).astype(np.int32)  # exclusivo

erros = []

# Redução ingênua
r_soma = reducao_ingenua(d_val, 'soma')
r_max  = reducao_ingenua(d_val, 'max')
r_min  = reducao_ingenua(d_val, 'min')
if r_soma != ref_sum: erros.append(f'Ingênua soma: esperado {ref_sum}, obtido {r_soma}')
if r_max  != ref_max: erros.append(f'Ingênua max: esperado {ref_max}, obtido {r_max}')
if r_min  != ref_min: erros.append(f'Ingênua min: esperado {ref_min}, obtido {r_min}')

# Hillis-Steele
hs_out = scan_hillis_steele(d_val)
if not np.array_equal(hs_out, ref_scan_inc):
    erros.append(f'Hillis-Steele: diferença em {(hs_out != ref_scan_inc).sum()} posições')

# Blelloch
bl_out = scan_blelloch(d_val)
if not np.array_equal(bl_out, ref_scan_exc):
    erros.append(f'Blelloch: diferença em {(bl_out != ref_scan_exc).sum()} posições')

if erros:
    print('FALHAS DE CORRETUDE — corrija antes de medir!')
    for e in erros:
        print(' ✗', e)
else:
    print('Todas as kernels estão corretas ✓')
    print(f'  Ingênua  → soma={r_soma}, max={r_max}, min={r_min}')
    print(f'  Hillis-Steele  → scan inclusivo OK (512 posições)')
    print(f'  Blelloch → scan exclusivo OK (512 posições)')

## Célula 9 — Funções utilitárias de benchmark

In [ ]:
# ── Utilitários de medição ────────────────────────────────────────────────────

def medir_cpu(fn, v, reps=REPETICOES):
    """Retorna (media_ms, desvio_ms) para função CPU."""
    tempos = []
    for _ in range(reps):
        t0 = time.perf_counter()
        fn(v)
        tempos.append((time.perf_counter() - t0) * 1000)
    tempos_corte = sorted(tempos)[1:-1]   # descarta min e max
    return float(np.mean(tempos_corte)), float(np.std(tempos_corte))

def medir_gpu(fn_gpu, v_gpu, reps=REPETICOES):
    """Retorna (media_ms, desvio_ms) usando cudaEvent (evento CUDA)."""
    tempos = []
    for _ in range(reps):
        start = cuda.event(timing=True)
        stop  = cuda.event(timing=True)
        start.record()
        fn_gpu(v_gpu)
        stop.record()
        stop.synchronize()
        tempos.append(cuda.event_elapsed_time(start, stop))
    tempos_corte = sorted(tempos)[1:-1]
    return float(np.mean(tempos_corte)), float(np.std(tempos_corte))

def speedup(t_cpu, t_gpu):
    return round(t_cpu / t_gpu, 4) if t_gpu > 0 else float('inf')

print('Utilitários de medição prontos.')

## Célula 10 — Benchmark: Redução (soma, máximo, mínimo)

In [ ]:
# ── Benchmark de redução ──────────────────────────────────────────────────────
# Compara CPU sequencial vs kernel ingênua GPU para soma, max e min

registros_reducao = []

ops_cpu = {'soma': soma_cpu, 'max': maximo_cpu, 'min': minimo_cpu}
ops_gpu = {'soma': lambda g: reducao_ingenua(g,'soma'),
           'max':  lambda g: reducao_ingenua(g,'max'),
           'min':  lambda g: reducao_ingenua(g,'min')}

for N in TAMANHOS:
    v_h = np.random.randint(0, 1000, size=N, dtype=np.int32)
    v_d = cuda.to_device(v_h)

    for op in ['soma', 'max', 'min']:
        t_cpu_m, t_cpu_s = medir_cpu(ops_cpu[op], v_h)
        t_gpu_m, t_gpu_s = medir_gpu(ops_gpu[op], v_d)
        sp = speedup(t_cpu_m, t_gpu_m)

        registros_reducao.append({
            'kernel': f'ingenua_{op}',
            'N': N,
            'threads_por_bloco': BLOCO_NAIVE,
            'tempo_cpu_ms': round(t_cpu_m, 4),
            'desvio_cpu_ms': round(t_cpu_s, 4),
            'tempo_gpu_ms': round(t_gpu_m, 4),
            'desvio_gpu_ms': round(t_gpu_s, 4),
            'speedup': sp,
        })
        print(f'N={N:>10,} | {op:4} | CPU {t_cpu_m:8.3f}ms | GPU {t_gpu_m:8.3f}ms | speedup {sp:.2f}x')

df_reducao = pd.DataFrame(registros_reducao)
print('\nBenchmark de redução concluído.')

## Célula 11 — Benchmark: Scan (Hillis-Steele vs Blelloch vs CPU)

In [ ]:
# ── Benchmark de scan ─────────────────────────────────────────────────────────
# Limita o tamanho máximo ao tamanho do bloco das kernels (512)
# Para entradas maiores, exibiria speedup negativo (kernels single-block)

TAMANHOS_SCAN = [N for N in TAMANHOS if N <= 512] + [512]
TAMANHOS_SCAN = sorted(set(TAMANHOS_SCAN))

registros_scan = []

for N in TAMANHOS_SCAN:
    v_h = np.random.randint(0, 100, size=N, dtype=np.int32)
    v_d = cuda.to_device(v_h)

    t_cpu_m, t_cpu_s = medir_cpu(scan_cpu, v_h)
    t_hs_m,  t_hs_s  = medir_gpu(scan_hillis_steele, v_d)
    t_bl_m,  t_bl_s  = medir_gpu(scan_blelloch, v_d)

    sp_hs = speedup(t_cpu_m, t_hs_m)
    sp_bl = speedup(t_cpu_m, t_bl_m)

    for kernel, t_gpu_m, t_gpu_s, sp in [
        ('hillis_steele', t_hs_m, t_hs_s, sp_hs),
        ('blelloch',      t_bl_m, t_bl_s, sp_bl),
    ]:
        registros_scan.append({
            'kernel': kernel,
            'N': N,
            'threads_por_bloco': HS_BLOCO,
            'tempo_cpu_ms': round(t_cpu_m, 4),
            'desvio_cpu_ms': round(t_cpu_s, 4),
            'tempo_gpu_ms': round(t_gpu_m, 4),
            'desvio_gpu_ms': round(t_gpu_s, 4),
            'speedup': sp,
        })

    print(f'N={N:>6,} | CPU {t_cpu_m:8.4f}ms | HS {t_hs_m:8.4f}ms ({sp_hs:.2f}x) | BL {t_bl_m:8.4f}ms ({sp_bl:.2f}x)')

df_scan = pd.DataFrame(registros_scan)
print('\nBenchmark de scan concluído.')

## Célula 12 — Exportar CSVs para resultados/

In [ ]:
# ── Exporta dados brutos ───────────────────────────────────────────────────────

df_reducao.to_csv('resultados/benchmark_reducao.csv', index=False)
df_scan.to_csv('resultados/benchmark_scan.csv',       index=False)

df_todos = pd.concat([df_reducao, df_scan], ignore_index=True)
df_todos.to_csv('resultados/benchmark_completo.csv',  index=False)

print('CSVs exportados para resultados/')
print(df_todos[['kernel','N','tempo_cpu_ms','tempo_gpu_ms','speedup']].to_string(index=False))

## Célula 13 — Gráfico 1: Tempo de execução × Tamanho de entrada

In [ ]:
# ── Gráfico de tempo: CPU vs kernels GPU (redução — soma) ─────────────────────

df_soma = df_reducao[df_reducao['kernel'] == 'ingenua_soma'].copy()

fig, ax = plt.subplots(figsize=(8, 5))

ax.plot(df_soma['N'], df_soma['tempo_cpu_ms'],
        marker='o', label='CPU sequencial', linewidth=2, color='#444441')
ax.errorbar(df_soma['N'], df_soma['tempo_gpu_ms'],
            yerr=df_soma['desvio_gpu_ms'],
            marker='s', label='GPU ingênua (soma)', linewidth=2,
            color='#7F77DD', capsize=4)

ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('Tamanho da entrada (N)', fontsize=12)
ax.set_ylabel('Tempo médio (ms)', fontsize=12)
ax.set_title('Tempo de execução: CPU × GPU — Redução (soma)', fontsize=13)
ax.legend()
ax.grid(True, which='both', linestyle='--', alpha=0.4)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('resultados/tempo_reducao_soma.png', dpi=150)
plt.show()
print('Gráfico salvo em resultados/tempo_reducao_soma.png')

## Célula 14 — Gráfico 2: Speedup × Tamanho de entrada

In [ ]:
# ── Gráfico de speedup para todas as operações de redução ────────────────────

cores = {'ingenua_soma': '#7F77DD', 'ingenua_max': '#1D9E75', 'ingenua_min': '#EF9F27'}
labels = {'ingenua_soma': 'GPU ingênua — soma',
          'ingenua_max':  'GPU ingênua — max',
          'ingenua_min':  'GPU ingênua — min'}

fig, ax = plt.subplots(figsize=(8, 5))

ax.axhline(y=1, color='#888780', linestyle='--', linewidth=1.2, label='Speedup = 1 (limiar)')

for kernel in ['ingenua_soma', 'ingenua_max', 'ingenua_min']:
    df_k = df_reducao[df_reducao['kernel'] == kernel]
    ax.plot(df_k['N'], df_k['speedup'],
            marker='o', label=labels[kernel],
            linewidth=2, color=cores[kernel])

ax.set_xscale('log')
ax.set_xlabel('Tamanho da entrada (N)', fontsize=12)
ax.set_ylabel('Speedup (T_CPU / T_GPU)', fontsize=12)
ax.set_title('Speedup GPU vs CPU — Redução ingênua', fontsize=13)
ax.legend()
ax.grid(True, which='both', linestyle='--', alpha=0.4)
ax.xaxis.set_major_formatter(ticker.FuncFormatter(lambda x,_: f'{int(x):,}'))

plt.tight_layout()
plt.savefig('resultados/speedup_reducao.png', dpi=150)
plt.show()
print('Gráfico salvo em resultados/speedup_reducao.png')

## Célula 15 — Gráfico 3: Comparação Hillis-Steele vs Blelloch vs CPU

In [ ]:
# ── Gráfico de tempo: scan CPU vs HS vs Blelloch ─────────────────────────────

df_hs = df_scan[df_scan['kernel'] == 'hillis_steele'].copy()
df_bl = df_scan[df_scan['kernel'] == 'blelloch'].copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Tempo
ax = axes[0]
ax.plot(df_hs['N'], df_hs['tempo_cpu_ms'],
        marker='o', label='CPU sequencial', linewidth=2, color='#444441')
ax.errorbar(df_hs['N'], df_hs['tempo_gpu_ms'], yerr=df_hs['desvio_gpu_ms'],
            marker='s', label='Hillis-Steele', linewidth=2, color='#7F77DD', capsize=4)
ax.errorbar(df_bl['N'], df_bl['tempo_gpu_ms'], yerr=df_bl['desvio_gpu_ms'],
            marker='^', label='Blelloch', linewidth=2, color='#1D9E75', capsize=4)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_xlabel('N', fontsize=11)
ax.set_ylabel('Tempo médio (ms)', fontsize=11)
ax.set_title('Tempo: scan CPU × GPU', fontsize=12)
ax.legend()
ax.grid(True, which='both', linestyle='--', alpha=0.4)

# Speedup
ax2 = axes[1]
ax2.axhline(y=1, color='#888780', linestyle='--', linewidth=1.2, label='Speedup = 1')
ax2.plot(df_hs['N'], df_hs['speedup'],
         marker='s', label='Hillis-Steele', linewidth=2, color='#7F77DD')
ax2.plot(df_bl['N'], df_bl['speedup'],
         marker='^', label='Blelloch', linewidth=2, color='#1D9E75')
ax2.set_xscale('log')
ax2.set_xlabel('N', fontsize=11)
ax2.set_ylabel('Speedup', fontsize=11)
ax2.set_title('Speedup: scan GPU vs CPU', fontsize=12)
ax2.legend()
ax2.grid(True, which='both', linestyle='--', alpha=0.4)

plt.tight_layout()
plt.savefig('resultados/benchmark_scan.png', dpi=150)
plt.show()
print('Gráfico salvo em resultados/benchmark_scan.png')

## Célula 16 — Tabela resumo formatada

In [ ]:
# ── Tabela resumo para o relatório ────────────────────────────────────────────

print('=== REDUÇÃO ===')
print(df_reducao[['kernel','N','tempo_cpu_ms','desvio_cpu_ms',
                   'tempo_gpu_ms','desvio_gpu_ms','speedup']]
      .to_string(index=False))

print('\n=== SCAN ===')
print(df_scan[['kernel','N','tempo_cpu_ms','desvio_cpu_ms',
               'tempo_gpu_ms','desvio_gpu_ms','speedup']]
      .to_string(index=False))

print('\n=== ARQUIVOS EM resultados/ ===')
for f in sorted(os.listdir('resultados')):
    tamanho = os.path.getsize(f'resultados/{f}')
    print(f'  {f:45s}  {tamanho:>8,} bytes')